# Basic

In [ ]:
%load_ext autoreload
%autoreload all

In [2]:
import polars as pl
import pickle

import src.graph_tokenizer_gd_tree_dev.config as config
import src.graph_tokenizer_gd_tree_dev.graph_fct as graph_fct


In [4]:
df_relations = pl.read_parquet(f"{config.BasicConfig().relation_path}")
df_mapped = pl.read_parquet(f"{config.BasicConfig().mapped_path}")


In [4]:
mapped_ids = df_mapped["id"].to_list()

# remove root concept
df_relations = df_relations.filter(~pl.col("dst.id").is_in(config.TokenizerParam().exclude_cpt))

# build graph and combine
whole_graph = graph_fct.build_relations_graph(df_relations, col_src="src.id", col_dst="dst.id", col_relation="relation")
combined_subgraphs = graph_fct.get_combined_subgraphs_from_nodes(whole_graph, mapped_ids, max_distance = config.TokenizerParam().max_dist_candidate)

with open(config.ProcessedGraph().combined_subgraphs, "wb") as f:
    pickle.dump(combined_subgraphs, f)

In [5]:
df_cpt = pl.read_parquet(config.BasicConfig().concept_path).select("id", "label")
id_to_label = dict(zip(df_cpt["id"], df_cpt["label"]))
with open(config.ProcessedGraph().id_to_label, "wb") as f:
    pickle.dump(id_to_label, f)